In [2]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import CountVectorizer
import random
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    doc = nlp(text)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and token.is_alpha
    ]
    return " ".join(tokens)

def generate_plot_from_word(vectorizer, bow_matrix, word, top_n=5, df_subset=None):
    cleaned_word = preprocess(word)
    word_vec = vectorizer.transform([cleaned_word])
    similarities = cosine_similarity(word_vec, bow_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_n]
    
    results = []
    for i in top_indices:
        title = df_subset.iloc[i]['Title']
        plot = df_subset.iloc[i]['Plot']
        results.append((title, plot))
    return results

def main():
    # Load the CSV file
    df = pd.read_csv('/kaggle/input/movie-plots/movieplots.csv')  # Replace with your actual file path
    df_subset = df.sample(frac=0.1, random_state=42).dropna(subset=['Plot'])  # Sample 10% for speed
    print("Subset of plots created")

    # Preprocess plots
    cleaned_plots = [preprocess(plot) for plot in df_subset['Plot']]
    print("Plots preprocessed")

    # Bag of Words Vectorization
    vectorizer = CountVectorizer()
    bow_matrix = vectorizer.fit_transform(cleaned_plots)
    print("BoW Vectorization Complete")

    # Optional: Inspect Vocabulary and Shape
    print("Vocabulary size:", len(vectorizer.vocabulary_))
    print("BoW matrix shape:", bow_matrix.shape)

    # Word frequency analysis
    word_counts = bow_matrix.toarray().sum(axis=0)
    vocab = vectorizer.get_feature_names_out()
    word_freq_df = pd.DataFrame({'word': vocab, 'count': word_counts})
    top_words = word_freq_df.sort_values(by='count', ascending=False)
    print(top_words.head(20))

    # Generate plots based on user word
    user_word = "kill"
    generated_results = generate_plot_from_word(vectorizer, bow_matrix, user_word, top_n=5, df_subset=df_subset)

    for i, (title, plot) in enumerate(generated_results, start=1):
        print(f"\nTop Plot {i} — Title: {title}\nPlot:\n{plot}\n{'='*60}")

if __name__ == "__main__":
    main()

Subset of plots created
Plots preprocessed
BoW Vectorization Complete
Vocabulary size: 37347
BoW matrix shape: (3488, 37347)
         word  count
11298    find   3832
17583    kill   3381
18549   leave   2952
19894     man   2951
33070    tell   2923
11019  father   2466
32718    take   2449
19230    love   2428
12764      go   2261
11915  friend   2157
34221     try   2156
6370     come   2147
27617  return   2034
7799      day   1987
20764    meet   1955
10902  family   1905
18812    life   1871
33562    time   1802
12528     get   1788
14807   house   1762

Top Plot 1 — Title: 24 Mani Neram
Plot:
Heroine Nalini gets killed by villain Sathyaraj and the hero Mohan takes oath to kill the villain within 24 hours.

Top Plot 2 — Title: Ten Dark Women
Plot:
A married television executive has many mistresses. Nine of the mistresses and his wife band together and plan to kill him. His wife tells him they are planning to kill him and they fake his death at a meeting of all ten women using a p